# Figure 2 — why the obvious approaches fail

Five strategies, twenty experiments each, one landscape, each as its own square panel (axes generic: x1, x2 -- this is about the shape of the search, not the specific chemistry). No panel titles or method notes on the plots themselves -- just the best result found and where it is. Each strategy also gets a GIF of its twenty points landing in the order they were actually acquired. One-factor-at-a-time, grid, a classical DOE (face-centred CCD + quadratic response surface), finite-difference gradient ascent, and random search. Random search is included because it is the honest baseline any BO result must beat; DOE is included because it is the strongest of the five, and the one BO has to beat in fig. 15.


In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "_shared"))
sys.path.insert(0, str(pathlib.Path.cwd() / "_shared"))
import numpy as np, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import style, gp as gpmod, landscape as land, doe as doemod
style.use_deck_style()
OUT = "../lecture_12_figures/generated"

In [ ]:

from matplotlib.animation import FuncAnimation, PillowWriter

style.SHOW_TEXT = True   # False -> drop even the best-result label and the
                         # x1/x2 axis labels, for ungrouping into pptx shapes

T, C, Z = land.mesh()
xstar, zstar = land.optimum()
print("true optimum:", np.round(xstar, 2), "yield", round(zstar, 1))
BUDGET = 20
rng = np.random.default_rng(0)


def best_of(pts):
    pts = np.asarray(pts)
    yv = land.yield_surface(pts)
    i = int(np.argmax(yv))
    return pts[i], yv[i], i


def base(ax):
    """Contour + true optimum, shared by every static and animated panel."""
    cs = ax.contourf(T, C, Z, levels=18, cmap="BuGn", alpha=0.9)
    ax.contour(T, C, Z, levels=8, colors="white", linewidths=0.4, alpha=0.6)
    ax.plot(*xstar, "*", ms=15, color=style.RED, mec="white", mew=0.8, zorder=8)
    ax.set_xlim(*land.BOUNDS[0]); ax.set_ylim(*land.BOUNDS[1])
    ax.set_box_aspect(1)
    style.xlabel(ax, "x1")
    style.ylabel(ax, "x2")
    return cs


def best_label(ax, pts):
    pts = np.asarray(pts)
    _, best, i = best_of(pts)
    style.text(ax, 0.03, 0.03, f"best {best:.0f}%\nfound at run {i + 1} of {len(pts)}",
               transform=ax.transAxes, fontsize=10, color=style.INK,
               va="bottom", fontweight="bold", zorder=10,
               bbox=dict(fc="white", alpha=0.92, ec="none", pad=2.2))


def static_plot(name, pts, colour, predict=None):
    """One square panel per strategy: contour, path, points, best-result label."""
    pts = np.asarray(pts)
    fig, ax = plt.subplots(figsize=(4.3, 4.3))
    cs = base(ax)
    if predict is not None:
        ax.contour(T, C, predict(T, C), levels=7, colors=colour, linewidths=0.9,
                  linestyles="--", alpha=0.9)
    ax.plot(pts[:, 0], pts[:, 1], "-", color=colour, lw=0.9, alpha=0.55)
    ax.scatter(pts[:, 0], pts[:, 1], s=32, c=colour, edgecolor="white",
              linewidth=0.7, zorder=6)
    best_label(ax, pts)
    cb = fig.colorbar(cs, ax=ax, fraction=0.046, pad=0.04)
    style.cbar_label(cb, "true yield / %")
    cb.outline.set_visible(False)
    style.save(fig, f"fig_02_{name}", OUT)
    plt.close(fig)


def animate(name, pts, colour):
    """GIF: the strategy's own twenty points landing in acquisition order."""
    pts = np.asarray(pts)
    fig_a, ax_a = plt.subplots(figsize=(4.3, 4.3))

    def _draw(k):
        ax_a.clear()
        base(ax_a)
        if k:
            shown = pts[:k]
            if k > 1:
                ax_a.plot(shown[:, 0], shown[:, 1], "-", color=colour, lw=0.8,
                         alpha=0.45)
                ax_a.scatter(shown[:-1, 0], shown[:-1, 1], s=30, c=colour,
                            edgecolor="white", linewidth=0.7, zorder=6)
            ax_a.scatter(shown[-1:, 0], shown[-1:, 1], s=60, c=style.GOLD,
                        edgecolor="white", linewidth=1.0, zorder=7)
            best_label(ax_a, shown)

    anim = FuncAnimation(fig_a, _draw, frames=range(len(pts) + 1), interval=400)
    gif_path = f"{OUT}/fig_02_{name}_acquisition.gif"
    anim.save(gif_path, writer=PillowWriter(fps=2.5))
    plt.close(fig_a)
    print("wrote", gif_path)


# --- 1. one factor at a time -------------------------------------------------
# walks up one axis, turns once, stops -- misses the curved ridge entirely
c0 = 2.0
ts = np.linspace(*land.BOUNDS[0], 10)
leg1 = [[t, c0] for t in ts]
tbest = ts[int(np.argmax(land.yield_surface(np.array(leg1))))]
cs_ = np.linspace(*land.BOUNDS[1], 10)
leg2 = [[tbest, c] for c in cs_]
ofat = np.array(leg1 + leg2)

# --- 2. grid ------------------------------------------------------------------
# regular, and mostly in bad regions -- 6 factors x 5 levels = 15,625 runs
gt = np.linspace(land.BOUNDS[0, 0] + 8, land.BOUNDS[0, 1] - 8, 5)
gc = np.linspace(land.BOUNDS[1, 0] + 0.4, land.BOUNDS[1, 1] - 0.4, 4)
grid = np.array([[a, b] for a in gt for b in gc])

# --- 3. DOE -- central composite design + quadratic RSM ----------------------
# a face-centred design + fitted quadratic (see _shared/doe.py and fig. 15)
res_doe = doemod.run(land.BOUNDS, land.yield_surface, BUDGET, seed=0)
doe_pts = res_doe["X"]

# --- 4. finite-difference gradient ascent ------------------------------------
# starts well off the peak, on the correct side of the deceptive bump so it
# climbs onto the ridge instead of getting trapped; d+1 runs per step means
# only ~6 real moves for the budget, so it still falls short of the optimum
NOISE = 1.0
p = np.array([95.0, 0.6])
path = [p.copy()]
step = np.array([18.0, 1.1])
h = np.array([4.0, 0.25])
while len(path) < BUDGET - 2:
    fp = land.f(p) + rng.normal(0, NOISE)
    g = np.zeros(2)
    for j in range(2):
        q = p.copy(); q[j] += h[j]
        q = np.clip(q, land.BOUNDS[:, 0], land.BOUNDS[:, 1])
        path.append(q.copy())
        g[j] = (land.f(q) + rng.normal(0, NOISE) - fp) / h[j]
    n = np.linalg.norm(g)
    if n < 1e-9:
        break
    p = np.clip(p + step * g / n, land.BOUNDS[:, 0], land.BOUNDS[:, 1])
    path.append(p.copy())
gradient = np.array(path)[:BUDGET]

# --- 5. random ----------------------------------------------------------------
# the honest baseline -- always compare against it
rand = rng.uniform(land.BOUNDS[:, 0], land.BOUNDS[:, 1], size=(BUDGET, 2))

STRATEGIES = [
    ("ofat", ofat, style.RED, None),
    ("grid", grid, style.INK, None),
    ("doe", doe_pts, style.PLUM, res_doe["predict"]),
    ("gradient", gradient, style.GOLD, None),
    ("random", rand, style.TEAL, None),
]

for name, pts, colour, predict in STRATEGIES:
    static_plot(name, pts, colour, predict)
    animate(name, pts, colour)